# 17 · Fabric IQ ontology and operations agents (PREVIEW)

## Goal

Wire the Fabric IQ MCP (Preview) tool for ontology lookups, and build a
write-path operation with explicit business-rule cascading — the point
where "the agent can act on data" stops being a slogan and starts being a
safety question.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("FABRIC_WORKSPACE_ID", "FABRIC_ONTOLOGY_ID")
print("PREVIEW surface — Fabric IQ MCP tool")


## Concept

Fabric IQ Ontology arrives as a first-party MCP (Preview) tool, needing a
Workspace ID and an Ontology ID; the tool exposes operations like
`list_ontology_entity_types` and `search_ontology`. That covers read.
Write-path operations are the harder half: an operational agent that can
*act* on data (updating a supplier's risk flag, say) needs the business
rules that cascade from that change modeled explicitly, not left to the
model's judgement in the moment — a flag change might need to notify
procurement, block auto-renewal, or both, and that cascade has to be
deterministic, which is why it's a workflow step calling the ontology tool,
not a bare tool call the agent makes freely.


## Build


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

mcp_servers = yaml.safe_load((workspace / "mcp-servers.yaml").read_text())
mcp_servers.append({
    "id": "fabric-iq-ontology-mcp",
    "transport": "streamable-http",
    "workspaceId": settings.get("FABRIC_WORKSPACE_ID"),
    "ontologyId": settings.get("FABRIC_ONTOLOGY_ID"),
    "auth": {"type": "on-behalf-of"},
    "toolset": ["list_ontology_entity_types", "search_ontology", "update_entity_risk_flag"],
})
(workspace / "mcp-servers.yaml").write_text(yaml.dump(mcp_servers, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### Write-path safety: cascading business rules as a workflow, not a bare tool call


In [ ]:
import yaml
from pathlib import Path
workflow_dir = Path("../agents/contract-renewal-desk/workflows")

flag_update_workflow = {
    "name": "supplier-risk-flag-update",
    "trigger": {"type": "manual"},
    "inputs": [{"name": "supplierName", "type": "string", "required": True},
               {"name": "riskFlag", "type": "boolean", "required": True}],
    "steps": [
        {"id": "updateFlag", "type": "mcp-tool-call", "server": "fabric-iq-ontology-mcp",
         "tool": "update_entity_risk_flag", "arguments": {"entity": "@{inputs.supplierName}", "flag": "@{inputs.riskFlag}"}},
        {"id": "cascadeCheck", "type": "if-else", "condition": "@{inputs.riskFlag} == true",
         "ifTrue": [
             {"id": "notifyProcurement", "type": "notification", "channel": "teams", "message": "Risk flag set on @{inputs.supplierName} — auto-renewal blocked pending review."},
             {"id": "blockAutoRenewal", "type": "dataverse-update", "entity": "crd_supplierrenewal", "field": "autoRenewalEnabled", "value": False},
         ],
         "ifFalse": []},
    ],
}
(workflow_dir / "supplier-risk-flag-update.yaml").write_text(yaml.dump(flag_update_workflow, sort_keys=False))

from csx.pac import copilot_push
copilot_push(Path("../agents/contract-renewal-desk"))


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["fabric-iq"]), credit_meter=meter, min_pass_rate=0.75)


## Cost


In [ ]:
meter.report_cost("17", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="ontology MCP wiring + cascading-rule workflow verification")


## Teardown


In [ ]:
print("No teardown — fabric-iq-ontology-mcp and the cascade workflow persist through 25.")
